In [ ]:
!pip install langgraph langchain openai


In [ ]:
!pip install -U langchain-openai


In [ ]:
!pip install --upgrade openai


In [4]:
import openai
print(openai.__version__) 


1.69.0


In [5]:
import langgraph as lg
dir(lg)


['__doc__',
 '__file__',
 '__loader__',
 '__name__',
 '__package__',
 '__path__',
 '__spec__']

In [6]:
import langgraph as lg

# List available attributes and methods in the LangGraph module
print(dir(lg))


['__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__']


In [7]:
# Print the attributes of LangGraph's 'lg' module
help(lg)

# Print the classes that LangGraph imports
import inspect
print(inspect.getmembers(lg, inspect.isclass))


Help on package langgraph:

NAME
    langgraph

PACKAGE CONTENTS
    _api (package)
    channels (package)
    config
    constants
    errors
    func (package)
    graph (package)
    managed (package)
    prebuilt (package)
    pregel (package)
    types
    utils (package)
    version

FILE
    (built-in)


[]


In [8]:
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages

class State(TypedDict):
    messages: Annotated[list, add_messages]

graph_builder = StateGraph(State)


In [ ]:
import os
from langchain_openai import AzureChatOpenAI
os.environ["OPENAI_API_VERSION"] = "2023-05-15"
os.environ["AZURE_OPENAI_API_KEY"] = "your_api_key"
os.environ["AZURE_OPENAI_ENDPOINT"] = "https://your-endpoint.openai.azure.com/"
customer_llm = AzureChatOpenAI()
project_manager_llm = AzureChatOpenAI()
requirements_engineer_llm = AzureChatOpenAI()
def customer(state):
    return {"messages": [customer_llm.invoke(state["messages"])]}

def project_manager(state):
    return {"messages": [project_manager_llm.invoke(state["messages"])]}

def requirements_engineer(state):
    return {"messages": [requirements_engineer_llm.invoke(state["messages"])]}



In [10]:
graph_builder.add_edge(START, "customer")
graph_builder.add_edge("customer", "project_manager")
graph_builder.add_edge("project_manager", "requirements_engineer")


In [ ]:
import os
os.environ["OPENAI_API_VERSION"] = "2023-05-15"
os.environ["AZURE_OPENAI_API_KEY"] = "your_api_key"
os.environ["AZURE_OPENAI_ENDPOINT"] = "https://your-endpoint.openai.azure.com/"


In [12]:
from langchain_openai import AzureChatOpenAI
from langchain_openai import AzureChatOpenAI
customer_llm = AzureChatOpenAI(api_version="2023-05-15")
project_manager_llm = AzureChatOpenAI(api_version="2023-05-15")
requirements_engineer_llm = AzureChatOpenAI(api_version="2023-05-15")
system_engineer_llm = AzureChatOpenAI(api_version="2023-05-15")
software_developer_llm = AzureChatOpenAI(api_version="2023-05-15")
test_engineer_llm = AzureChatOpenAI(api_version="2023-05-15")
documentation_engineer_llm = AzureChatOpenAI(api_version="2023-05-15")


In [13]:
def customer(state: State):
    if not state["messages"]:
        return {"messages": ["Initial message for customer"]}
    return {"messages": [customer_llm.invoke(state["messages"])]}


In [14]:
import time
import random

def invoke_with_retry(llm, input, max_retries=10, initial_delay=1, exponential_base=2):
    retries = 0
    delay = initial_delay
    while retries < max_retries:
        try:
            return llm.invoke(input)
        except Exception as e:
            print(f"Error invoking LLM: {e}. Retrying in {delay} seconds...")
            time.sleep(delay + random.random())  # Add some jitter to the delay
            delay *= exponential_base
            retries += 1
    raise Exception("Failed after retries.")


In [ ]:
import openai
openai.api_key = "your_api_key"


In [ ]:
from autogen import OpenAIWrapper

client = OpenAIWrapper(
    config_list=[{"api_key": "your_api_key"}]
)


In [ ]:
import openai
import langgraph as lg
import autogen
import os
api_key = os.getenv("OPENAI_API_KEY", "")
if not api_key:
    raise ValueError("Please set your OpenAI API key in an environment variable or directly in the script.")

llm_config = {
    "model": "gpt-4-turbo",
    "api_key": api_key
}

customer_proxy = autogen.ConversableAgent(
    name="Customer",
    system_message=(
         
        " A customer providing requirements for an educational portal to the Project Manager."
    ),
    code_execution_config=False,
    llm_config=llm_config,
    human_input_mode="NEVER",
)

project_manager = autogen.ConversableAgent(
    name="Project_Manager",
    system_message=(
        
          """You are a project manager coordinating the development of an educational portal. Your responsibilities include:",
    1. Receiving requirements from the customer
    2. Delegating tasks to appropriate team members (Requirement Engineer, System Engineer, Software Engineer, Test Engineer, and Document Enginner)
    3. Collecting outputs from each engineer
    4. Compiling a comprehensive project plan with timelines after gathering estimated effort from each engineer.
    5. Presenting the final project plan in a tabular form to the customer"""
    ),
    code_execution_config=False,
    llm_config=llm_config,
    human_input_mode="NEVER",
)

requirements_engineer = autogen.ConversableAgent(
    name="Requirements_Engineer",
    system_message=(
       """ "You specialize in educational software requirements engineering.\n"
        "- Draft a detailed requirements document.\n"
        "- Provide realistic effort estimations."
        Total requirements: Estimate based on feature complexity
    - Productivity: 5 requirements per day
    - Effort = Total requirements / Productivity
    5. Return the requirements document and effort estimate to the project manager
    Be thorough and consider all aspects of the educational portal
        """
    ),
    code_execution_config=False,
    llm_config=llm_config,
    human_input_mode="NEVER",
)

system_engineer = autogen.ConversableAgent(
    name="System_Engineer",
    system_message=(
        """ "You design system architecture and APIs based on requirements.\n"
        "- Provide detailed system design and database schema.\n"
        "- Estimate implementation effort."
        Estimate effort based on:
    - Total design pages: 1 page per 5 requirements
    - Productivity: 5 pages per day
    - Effort = Total pages / Productivity"""
    ),
    code_execution_config=False,
    llm_config=llm_config,
    human_input_mode="NEVER",
)

software_developer = autogen.ConversableAgent(
    name="Software_Developer",
    system_message=(
      """ You are a software developer working on an educational software. Your tasks are:
    1. Receive design document from project manager
    2. Develop the application including:
    - Frontend components
    - Backend services
    - Database integration
    3. Estimate effort based on:
    - Lines of code: 100 SLOC per design page
    - Productivity: 50 SLOC per day
    - Effort = Total SLOC / Productivity
    4.Return effort estimate to project manager """
    ),
    code_execution_config=False,
    llm_config=llm_config,
    human_input_mode="NEVER",
)

test_engineer = autogen.ConversableAgent(
    name="Test_Engineer",
    system_message=(
       """ "You create test plans and conduct testing.\n"
        "- Perform unit and integration tests.\n"
        "- Estimate testing effort."
        Estimate effort based on:
    - Test cases: 2 test cases per requirement
    - Productivity: 2 test cases per day (including execution)
    - Effort = Total test cases / Productivity
    Return test plan and effort estimate to project manager"""
    ),
    code_execution_config=False,
    llm_config=llm_config,
    human_input_mode="NEVER",
)

documentation_engineer = autogen.ConversableAgent(
    name="Documentation_Engineer",
    system_message=(
       """ "You are responsible for writing documentation.\n"
        "- Create user manuals, API guides, and admin instructions.\n"
        "- Provide effort estimates."
        Estimate effort based on:
    - Documentation pages: 1 page per 3 requirements
    - Productivity: 3 pages per day
    - Effort = Total pages / Productivity"""
    ),
    code_execution_config=False,
    llm_config=llm_config,
    human_input_mode="NEVER",
)
groupchat = autogen.GroupChat(
    agents=[
        customer_proxy,
        project_manager,
        requirements_engineer,
        system_engineer,
        software_developer,
        test_engineer,
        documentation_engineer,
    ],
    messages=[],
    speaker_selection_method="round_robin",
    max_round=14,
)

manager = autogen.GroupChatManager(
    groupchat=groupchat,
    llm_config=llm_config
)

customer_statement = """
I want to create an educational portal where students and teachers can interact.
The portal should include features such as course management, assignment submission, grading system, 
and discussion forums. Teachers should be able to create and manage courses, while students should be able
to enroll, submit assignments, and receive feedback. The system should also support document sharing and 
administrative controls.
"""

chat_result = customer_proxy.initiate_chat(
    manager,
    message=customer_statement,
)

print(chat_result)


Customer (to chat_manager):


I want to create an educational portal where students and teachers can interact.
The portal should include features such as course management, assignment submission, grading system, 
and discussion forums. Teachers should be able to create and manage courses, while students should be able
to enroll, submit assignments, and receive feedback. The system should also support document sharing and 
administrative controls.


--------------------------------------------------------------------------------

Next speaker: Project_Manager

Project_Manager (to chat_manager):

Thank you for providing the requirements for the educational portal. I will now proceed to delegate tasks to the appropriate team members based on these requirements:

1. **Requirement Engineer**
   - Gather detailed requirements and clarify any ambiguities in the current requirements.
   - Verify the scope of features such as course management, assignment submission, grading system, discussion 